# 🗣️ Notebook 1: What is a Gossip Protocol?

**The big question:** *"In a cluster of 1000 nodes, how do you tell everyone something — without sending 1000 messages from one place?"*

A naïve answer: have one "leader" broadcast to all 999 others. That works for 10 nodes. At 10,000 nodes it falls over: the leader is a bottleneck and a **single point of failure** (SPOF). If the leader crashes mid-broadcast, half the cluster never hears the news.

A **gossip protocol** copies what humans do at a party: every few seconds, each node picks a small random set of peers and tells them what it knows. Those peers do the same. Information spreads exponentially — like a rumour, or a virus (which is why this family of algorithms is also called **epidemic protocols**).

In this notebook we compare:

1. 🟥 **BAD** — central broadcaster, one node tells everyone.
2. 🟩 **GOOD** — each node tells `fanout=3` random peers per round.

We measure: *how many rounds to reach every node*, *how many messages were sent in total*, and *what happens when the network drops messages.*

## Learning objectives
- Understand the epidemic / push-gossip algorithm.
- Observe `O(log N)` rounds to converge regardless of cluster size.
- See why gossip uses more *total* messages but no *bottleneck* node.
- Watch gossip stay reliable even when ~30% of messages are lost.

## 🛠️ Setup

```bash
cd 02-distributed-primitives/gossip-protocol
uv sync
```

Select the `.venv` kernel in VS Code (top-right of the notebook). If it's missing, reload the window with `Cmd+Shift+P` → "Reload Window".

In [ ]:
import random
import math
random.seed(0)

N = 100             # nodes in the cluster
FANOUT = 3          # peers each node tells per round

## 🟥 Approach 1: Central broadcaster (the BAD way)

One node tries to tell all the others by itself. To make it concrete (and a bit generous) we say it has limited bandwidth and can fan out 3 messages per round.

Two problems jump out:

1. **Linear time.** It needs `(N-1)/3` rounds — grows with the cluster.
2. **SPOF.** If the broadcaster dies after round 2, every node it hadn't reached yet is silently uninformed forever.

In [ ]:
def central_broadcast(n, msgs_per_round=3):
    informed = {0}     # node 0 starts with the news
    rounds = 0
    msgs_sent = 0
    while len(informed) < n:
        rounds += 1
        candidates = [i for i in range(n) if i not in informed]
        for target in candidates[:msgs_per_round]:
            informed.add(target)
            msgs_sent += 1
    return rounds, msgs_sent

rounds, msgs = central_broadcast(N)
print(f"central broadcast: {rounds} rounds, {msgs} messages, all from node 0")

## 🟩 Approach 2: Push gossip (the GOOD way)

Each round, **every informed node** picks `FANOUT` random peers and shares the news. Information multiplies by ~`fanout` every round, so it grows exponentially.

We also count how many messages were sent so we can compare cost against the broadcaster.

In [ ]:
def gossip(n, fanout=3, loss=0.0, rng=None):
    """Simple push gossip.

    - n: number of nodes
    - fanout: peers each informed node contacts per round
    - loss: probability that any single gossip message is dropped (0..1)
    - rng: optional random.Random for reproducibility
    """
    rng = rng or random
    informed = {0}
    history = [len(informed)]
    msgs_sent = 0
    rounds = 0
    # Cap rounds so a pathological run (e.g. loss=1.0) can't loop forever.
    max_rounds = 10 * max(1, int(math.log2(n))) + 50
    while len(informed) < n and rounds < max_rounds:
        rounds += 1
        new_informed = set(informed)
        for src in informed:
            for _ in range(fanout):
                peer = rng.randrange(n)
                msgs_sent += 1
                if rng.random() < loss:
                    continue          # message dropped by the network
                new_informed.add(peer)
        informed = new_informed
        history.append(len(informed))
    return rounds, msgs_sent, history

rounds, msgs, history = gossip(N, FANOUT)
print(f"gossip: {rounds} rounds, {msgs} messages, fanout={FANOUT}")
print("informed per round:", history)

### 💡 Trade-off in one line

Gossip uses **more** total messages than a central broadcaster, but no single node is the bottleneck and no single node is critical. Each node sends only `fanout` messages per round — a constant, no matter how big the cluster gets.

## 📈 Convergence shape: the S-curve

The number of informed nodes follows a classic *epidemic* S-curve: slow start (only one node knows), then explosive growth, then a long tail as the last few stragglers hear the rumour.

In [ ]:
import matplotlib.pyplot as plt

# Run gossip for several cluster sizes to show O(log N) scaling.
sizes = [50, 100, 500, 1000, 5000]
for n in sizes:
    _, _, h = gossip(n, FANOUT)
    plt.plot(range(len(h)), [x / n for x in h], label=f"N={n}")
plt.xlabel("round")
plt.ylabel("fraction of nodes informed")
plt.title(f"Gossip convergence — S-curve (fanout={FANOUT})")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

### 🧮 Why `O(log N)` rounds?

If every informed node tells `f` peers each round, the count of informed nodes roughly multiplies by `(1 + f)` per round (until saturation). So to cover `N` nodes you need about `log_{1+f}(N)` rounds. Doubling `N` adds **just one more round** when `f=1`, and even less for higher fanouts. That's why gossip scales: the number of rounds barely grows.

In [ ]:
# How does fanout affect convergence on a 1000-node cluster?
# Average over several seeds so one unlucky run doesn't drive the conclusion.
def mean_rounds(n, fanout, seeds=range(15)):
    return sum(gossip(n, fanout, rng=random.Random(s))[0] for s in seeds) / len(list(seeds))

print(f"{'fanout':>6}  {'mean rounds':>11}  {'log_{1+f}(N)':>12}")
measured = {}
for f in (1, 2, 3, 5, 10):
    measured[f] = mean_rounds(1000, f)
    print(f"{f:>6d}  {measured[f]:>11.1f}  {math.log(1000, 1 + f):>12.2f}")

# More fanout must never make convergence slower. Prove it, don't claim it.
fanouts = sorted(measured)
assert all(measured[a] > measured[b] for a, b in zip(fanouts, fanouts[1:])), measured

# And convergence must scale like log N, not like N: a 100x bigger cluster
# needs only a couple more rounds, not 100x more.
print()
print(f"{'N':>6}  {'mean rounds (fanout=3)':>22}")
by_n = {}
for n in (50, 500, 5000):
    by_n[n] = mean_rounds(n, 3)
    print(f"{n:>6d}  {by_n[n]:>22.1f}")
assert by_n[5000] < by_n[50] + 8, by_n          # 100x the nodes, a handful more rounds
assert by_n[5000] / by_n[50] < 3, by_n          # nowhere near linear (that would be 100x)
print("\n✔ measured: rounds fall with fanout and grow like log N, not like N")


## 🌐 What if the network drops messages?

Real networks lose packets. A central broadcaster's lost message is lost forever (the receiver may never get told). Gossip is naturally robust: every round, *many* nodes try to spread the news to *random* peers, so the same node will likely be re-told a few rounds later.

Below we replay the same gossip with 0%, 10%, 30%, and 50% message loss. Notice that even at 30% loss, convergence is only slightly slower.

In [ ]:
for loss in (0.0, 0.1, 0.3, 0.5):
    rng = random.Random(42)        # same seed for fairness
    rounds, _, h = gossip(1000, FANOUT, loss=loss, rng=rng)
    plt.plot(range(len(h)), [x / 1000 for x in h], label=f"loss={int(loss*100)}%")
    print(f"loss={loss:.0%}  ->  {rounds} rounds, final coverage {h[-1]}/1000")
plt.xlabel("round")
plt.ylabel("fraction informed")
plt.title("Push gossip is robust to message loss (N=1000, fanout=3)")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

# Robustness is a claim we can check: every loss level still reaches ALL 1000 nodes,
# and 30% loss costs us only a few extra rounds — not a failure to converge.
rounds_by_loss = {}
for loss in (0.0, 0.1, 0.3, 0.5):
    r, _, h = gossip(1000, FANOUT, loss=loss, rng=random.Random(42))
    rounds_by_loss[loss] = r
    assert h[-1] == 1000, f"loss={loss} failed to reach every node: {h[-1]}/1000"
assert rounds_by_loss[0.3] <= 2 * rounds_by_loss[0.0], rounds_by_loss
print("✔ full coverage at every loss level; 30% loss costs",
      rounds_by_loss[0.3] - rounds_by_loss[0.0], "extra rounds")


## 🌍 Where this is used in the real world

Gossip is the backbone of cluster-membership and failure-detection in many production systems. A few:

| System | What it gossips |
| --- | --- |
| **Apache Cassandra** | node up/down state, schema version, token ranges, generation numbers (covered in notebook 2) |
| **Amazon Dynamo / DynamoDB** | ring membership, key-range ownership; uses **seed nodes** to bootstrap |
| **HashiCorp Consul / Serf** | implements the **SWIM** gossip variant for membership and failure detection |
| **Redis Cluster** | every node gossips a small subset of the cluster bus messages |
| **Bitcoin / Ethereum** | transactions and blocks propagate via gossip on the peer-to-peer network |

These systems all want the same thing: a leaderless, scalable, lossy-network-tolerant way to keep cluster knowledge in sync.

## ✅ Recap

- **Central broadcaster** scales linearly *and* is a single point of failure.
- **Push gossip** converges in roughly `O(log_{1+fanout} N)` rounds. Doubling cluster size adds barely any rounds.
- Gossip uses **more total messages** but spreads the load over the whole cluster — no bottleneck, no SPOF.
- Gossip is **robust to message loss** because every round, many random retries happen in parallel.
- Real systems (Cassandra, Consul, Serf, DynamoDB, Redis Cluster, Bitcoin) all use gossip variants for membership and failure dissemination.

**Next →** [`02_push_pull_and_failure_detection.ipynb`](./02_push_pull_and_failure_detection.ipynb) — three flavors of gossip (push, pull, push-pull), and how nodes use gossip to detect that other nodes have died.